# Download and prep NISAR data

This shows how the NISAR data is downloaded and assembled into an Xarray data cube. 

In [ ]:
from shapely.geometry import box 
import py3dep 
import rasterio

import sys
import os
from pathlib import Path

# import from this repo 
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2
from scripts.download import download_nisar, build_cube, download_nisar_gslc
from scripts.layers import get_nlcd_layers 

First, we setup logging to get useful information and error messages. 

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True 
)

The track, frame, start and end dates are all known based on out study area. 

In [1]:
track = 19
frame = 21
start = '2025-12-01'
end = '2026-04-30'
proc_code = 'P05023'

data_dir = "../data/"
area_file = data_dir + "aso/er_20260318/ASO_EastRiver_2026Mar18.swe_50m.tif"

with rasterio.open(area_file) as src:
    bounds = src.bounds         
    crs = src.crs
shp = gpd.GeoDataFrame(geometry=[box(*bounds)], crs=crs)

out = data_dir + "nisar/east_river"
layers = ['unwrapped_phase', 'coherence_unwrapped', # 80 m interferogram 
            'wrapped_phase', 'coherence_wrapped',  # 20 m interferogram 
            'incidence_angle', 'perpendicular_baseline'] # for spatial coherence  

gslc_layers = ['backscatter', 'NEBZ'] # for thermal coherence 

NameError: name 'rasterio' is not defined

A function in scripts/download.py does the heavy lifting. If the files already exist, they'll be skipped. 

In [ ]:
###################### downloads nisar data #######################
(grans, files) = download_nisar(
    track=track,
    frame=frame,
    start_date=start,
    end_date=end,
    aoi=shp,
    layers=layers,
    output_dir=out,
    crid=proc_code
)

###################### downloads nisar data #######################
(grans, files) = download_nisar_gslc(
    track=track,
    frame=frame,
    start_date=start,
    end_date=end,
    aoi=shp,
    layers=gslc_layers,
    output_dir=out,
    crid=proc_code
)

Once it's downloaded, we can get a list of all the files that we have. 

In [ ]:
files = [item.resolve() for item in Path(data_dir + 'nisar/east_river/').rglob('*')]

Build some data cubes from the tif files. 

In [ ]:

####################### builds coherence xarrays #######################
wrapped_files = [f for f in files if "coherence_wrapped" in str(f)]
unwrapped_files = [f for f in files if "coherence_unwrapped" in str(f)]
inc_files = [f for f in files if "incidence_angle" in str(f)]

nisar_20m = build_cube(wrapped_files, "coherence")     # 20 m grid
nisar_80m = build_cube(unwrapped_files, "coherence")  # 80 m grid
nisar_inc = build_cube(inc_files, "inc_angle")


Next, add a DEM and NLCD canopy cover layer. 

In [ ]:
####################### adds nlcd and dem data #######################
# --- Wrapped dataset (20 m grid) ---
crs_20m = nisar_20m.rio.crs
aoi_20m = box(*nisar_20m.rio.bounds())

nlcd_20m = get_nlcd_layers(tile_aoi=aoi_20m, crs=crs_20m, tile_ref_grid=nisar_20m)

dem_20m = py3dep.get_dem(geometry=aoi_20m, resolution=20, crs=crs_20m)
dem_20m = dem_20m.rio.write_crs(dem_20m.rio.crs).rio.reproject_match(nisar_20m)

nisar_20m = nisar_20m.merge(nlcd_20m)
nisar_20m['elevation'] = dem_20m

# --- Unwrapped dataset (80 m grid) ---
crs_80m = nisar_80m.rio.crs
aoi_80m = box(*nisar_80m.rio.bounds())

nlcd_80m = get_nlcd_layers(tile_aoi=aoi_80m, crs=crs_80m, tile_ref_grid=nisar_80m)

dem_80m = py3dep.get_dem(geometry=aoi_80m, resolution=80, crs=crs_80m)
dem_80m = dem_80m.rio.write_crs(dem_80m.rio.crs).rio.reproject_match(nisar_80m)

nisar_80m = nisar_80m.merge(nlcd_80m)
nisar_80m['elevation'] = dem_80m

Save the zarrs. 

In [ ]:
####################### saves zarrs #######################
nisar_20m.to_zarr(data_dir + 'nisar/zarrs/coherence_20m', mode="w")
nisar_80m.to_zarr(data_dir + 'nisar/zarrs/coherence_80m', mode="w")

Plot the data, so we know what we are working with. 

In [ ]:
nisar_80m['coherence'].plot(col='pair', col_wrap=5, cmap='inferno')
nisar_20m['coherence'].plot(col='pair', col_wrap=5, cmap='inferno')